In [1]:
from dotenv import load_dotenv
load_dotenv('env')

True

In [2]:
import os
import re
import torch
from transformers import AutoTokenizer
from datasets import load_dataset
from tqdm.auto import tqdm

from io import StringIO
from contextlib import redirect_stdout
from time import perf_counter
from huggingface_hub import HfFileSystem, hf_hub_download
import jsonlines
from vllm import LLM, SamplingParams
import pandas as pd
from typing import Tuple, Optional
from collections import Counter
from copy import deepcopy

tqdm.pandas()

In [3]:
MODEL_ID = "Hrushi/AIMO-SFT-DDP-Merged-Deepseek-Math-7B-Base-N141128-E1-R128-A256"

In [4]:
%%time
llm = LLM(
    model=MODEL_ID,
    tokenizer=MODEL_ID,
    # kv_cache_dtype='fp8',
    dtype='float16',
    gpu_memory_utilization=0.95,
    tensor_parallel_size=torch.cuda.device_count(),
    # trust_remote_code=True,
    # adapter_name_or_path=LORA_ADAPTER_ID,
    swap_space=48,
    enable_prefix_caching=True,
    disable_log_stats=True,
    # enable_chunked_prefill=True,
    # max_num_batched_tokens=8192,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

WARNING 06-10 10:43:11 config.py:427] Possibly too large swap space. 48.00 GiB out of the 83.48 GiB total CPU memory is allocated for the swap space.
INFO 06-10 10:43:11 llm_engine.py:161] Initializing an LLM engine (v0.4.3) with config: model='Hrushi/AIMO-SFT-DDP-Merged-Deepseek-Math-7B-Base-N141128-E1-R128-A256', speculative_config=None, tokenizer='Hrushi/AIMO-SFT-DDP-Merged-Deepseek-Math-7B-Base-N141128-E1-R128-A256', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, rope_scaling=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=4096, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), seed=0, served_model_name=Hrushi/AIMO-SFT-DDP-Merged-Deepseek-Math-7B-Base-N141128-E1-R128-A256)


/opt/conda/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


INFO 06-10 10:43:12 weight_utils.py:207] Using model weights format ['*.safetensors']
INFO 06-10 10:43:16 model_runner.py:146] Loading model weights took 12.8725 GB
INFO 06-10 10:43:17 gpu_executor.py:83] # GPU blocks: 3177, # CPU blocks: 6553
INFO 06-10 10:43:46 model_runner.py:854] Capturing the model for CUDA graphs. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.
INFO 06-10 10:43:46 model_runner.py:858] CUDA graphs can take additional 1~3 GiB memory per GPU. If you are running out of memory, consider decreasing `gpu_memory_utilization` or enforcing eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.
INFO 06-10 10:43:53 model_runner.py:924] Graph capturing finished in 6 secs.
INFO 06-10 10:43:53 block_manager_v1.py:247] Automatic prefix caching is enabled.


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


CPU times: user 18.3 s, sys: 28.1 s, total: 46.4 s
Wall time: 42.8 s


## Data Download

In [5]:
def get_final_solution(x):
    sols = re.findall(r'\\boxed\{(.*?)\}', x, re.DOTALL)

    if len(sols) == 0:
        sols = re.findall(r'\\boxed (.*?)\$', x, re.DOTALL)
    
    # Check if all are same
    try:
        if all(sol == sols[0] for sol in sols):
            return sols[0]
        else:
            return sols[-1]
    except:
        print(sols)
        print(x)
        raise


In [6]:
MATH_ds = load_dataset('hendrycks/competition_math', trust_remote_code=True)['test']
MATH_df = MATH_ds.to_pandas()

MATH_df['final_answer'] = MATH_df['solution'].progress_apply(get_final_solution)
MATH_df['final_answer_is_numeric'] = MATH_df['final_answer'].progress_apply(lambda x: x.isnumeric())

MATH_numeric_df = MATH_df[MATH_df['final_answer_is_numeric']].copy(deep=True).reset_index(drop=True)
MATH_numeric_df['ExpID'] = [f"Exp{idx:0>4}" for idx in range(len(MATH_numeric_df))]
MATH_numeric_df

  0%|          | 0/5000 [00:00<?, ?it/s]

  0%|          | 0/5000 [00:00<?, ?it/s]

,problem,level,type,solution,final_answer,final_answer_is_numeric,ExpID
0,How many vertical asymptotes does the graph of...,Level 3,Algebra,The denominator of the rational function facto...,2,True,Exp0000
1,What is the positive difference between $120\%...,Level 1,Algebra,One hundred twenty percent of 30 is $120\cdot3...,10,True,Exp0001
2,"If $2^8=4^x$, what is the value of $x$?",Level 1,Algebra,Rewrite $4$ as $2^2$ to find $4^x=2^{2x}$. Si...,4,True,Exp0002
3,What is the 100th term of the arithmetic seque...,Level 2,Algebra,"The common difference is $10 - 6 = 4$, so the ...",402,True,Exp0003
4,Mr. Madoff invests 1000 dollars in a fund that...,Level 4,Algebra,Let $r$ be the annual interest rate. Then aft...,7,True,Exp0004
...,...,...,...,...,...,...,...
2878,Given $\|\mathbf{v}\| = 5$ and $\|\mathbf{w}\|...,Level 3,Precalculus,Note that\n\begin{align*}\n\operatorname{proj}...,5,True,Exp2878
2879,If $0^\circ < x < 180^\circ$ and $\cos x + \si...,Level 5,Precalculus,"From the given equation, $\cos x = \frac{1}{2}...",14,True,Exp2879
2880,"Let $x_1,$ $x_2,$ $x_3,$ $y_1,$ $y_2,$ and $y_...",Level 5,Precalculus,"In general,\n\[\frac{1}{2} \begin{vmatrix} x_1...",144,True,Exp2880
2881,Compute\n\[\frac{1}{\cos^2 10^\circ} + \frac{1...,Level 4,Precalculus,We can write\n\begin{align*}\n\frac{1}{\cos^2 ...,12,True,Exp2881


## Helper Functions

In [7]:
import signal
import functools
from time import sleep

class TimeoutExpired(Exception):
  """Custom exception for timeout"""
  pass

def timeout(seconds=10, error_message='Timeout'):
  """Decorator to timeout a function after a certain number of seconds.

  Args:
      seconds: The maximum number of seconds allowed for the function to run.
      error_message: The message to raise in case of timeout.

  Returns:
      The result of the decorated function if it finishes within the timeout,
      otherwise raises TimeoutExpired with the provided error message.
  """

  def decorator(func):
    @functools.wraps(func)  # Preserve function metadata
    def wrapper(*args, **kwargs):
      def _handle_timeout(signum, frame):
        raise TimeoutExpired(error_message)

      # Set the signal handler and a timer
      signal.signal(signal.SIGALRM, _handle_timeout)
      signal.alarm(seconds)

      try:
        result = func(*args, **kwargs)
      finally:
        # Cancel the alarm if the function finishes before timeout
        signal.alarm(0)
      return result
    return wrapper
  return decorator

In [8]:
@timeout(seconds=60, error_message="Code took too long!")
def run_code_v2(code:str) -> Tuple[str, Optional[str], Optional[str]]:
    with redirect_stdout(StringIO()) as f:
        try:
            exec(code)
            error_name = None
            error_msg = None
        except Exception as e:
            error_name = str(e.__class__.__name__)
            error_msg = str(e)
    return f.getvalue().strip(), error_name, error_msg

In [9]:
def get_code(output_str:str) -> str:
    return re.search(r"<code_block>(.*?)</code_block>", output_str, re.DOTALL).group(1).strip()

In [10]:
def get_final_answer(output_str:str) -> str:
    return re.search(r"<final_answer>(.*?)</final_answer>", output_str, re.DOTALL).group(1).strip()

In [11]:
def post_processing_v2(raw_output:str, input_text:str, end_reason:str):
    error_name, error_msg = None, None
    
    if end_reason == '</final_answer>':
        final_answer = get_final_answer(raw_output)
        input_text += raw_output
    else:
        final_answer = None
    
    if end_reason == '<code_output_block>':
        # Get code block
        try:
            code = get_code(raw_output)
            # Runt the code
            code_output, error_name, error_msg = run_code_v2(code)
            input_text += f"{raw_output}{code_output}</code_output_block>\n"
        except AttributeError:
            error_name = "CodeBlockSyntanxError"
            error_msg = None
            input_text += raw_output
        
    elif end_reason == "length":
        input_text += raw_output
        
    return input_text, final_answer, error_name, error_msg

## Problem Solving

In [12]:
sampling_params = SamplingParams(
    n=5,
    temperature=0.2,
    top_p=1,
    top_k=100,
    max_tokens=1024,
    skip_special_tokens=False,
    spaces_between_special_tokens=False,
    stop=['<code_output_block>', '</final_answer>'],
    include_stop_str_in_output=True
)

system_prompt = """Your are a high school student appearing for your math exam.
You will be given a math problem to solve, and your job is to write the detailed step by step solution, with good mathematical reasoning and using python code for calculations, simplifications, solving equations, etc.

Format help and instructions:
- Problem has been given enclosed in <math_problem></math_problem> tags.
- Every step must be written within <start_of_step><end_of_step> tags.
- Code needs to be always written inside <code_block></code_block> tags - No backticks or triple backticks.
- Code output will be given in <code_output_block></code_output_block> tags

Every calculation should be performed using writing supporting python code."""

In [13]:
idx = 1231
problem = MATH_numeric_df.problem[idx]
ofa = MATH_numeric_df.final_answer[idx]
print(f"Problem: {problem}")
print(f"Original Final Answer: {ofa}")

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "problem", "content": problem}
]

input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True).replace(tokenizer.bos_token, '')
print(input_text)

Problem: How many square units are in the area of the largest square that can be inscribed in a circle with radius 1 unit?
Original Final Answer: 2
Your are a high school student appearing for your math exam.
You will be given a math problem to solve, and your job is to write the detailed step by step solution, with good mathematical reasoning and using python code for calculations, simplifications, solving equations, etc.

Format help and instructions:
- Problem has been given enclosed in <math_problem></math_problem> tags.
- Every step must be written within <start_of_step><end_of_step> tags.
- Code needs to be always written inside <code_block></code_block> tags - No backticks or triple backticks.
- Code output will be given in <code_output_block></code_output_block> tags

Every calculation should be performed using writing supporting python code.

<math_problem>How many square units are in the area of the largest square that can be inscribed in a circle with radius 1 unit?</math_pr

In [14]:
nattempted = 0
ncorrect = 0
nwrong = 0
ncorrect_top3 = 0

for idx in tqdm(MATH_numeric_df.index.values):
    exp_id = MATH_numeric_df.ExpID[idx]
    problem = MATH_numeric_df.problem[idx]
    ofa = str(MATH_numeric_df.final_answer[idx])
    nattempted += 1
    
    print(f"Exp-ID: {exp_id}")
    print(f"Problem: {problem}")
    print(f"Original Final Answer: {ofa}")
    
    save_fpath = os.path.join('MATH_Inf_Outputs', f'{exp_id}.jsonl')
    if os.path.exists(save_fpath):
        sol_df = pd.read_json(save_fpath, lines=True, orient='records')
        final_answers = sol_df['final_answer'].values.tolist()
        final_answers = [str(x) for x in final_answers]
    
    else:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "problem", "content": problem}
        ]

        input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True).replace(tokenizer.bos_token, '')
        open_threads = [input_text]
        final_answers = []
        closed_threads = []
        rounds = 1

        while len(open_threads) > 0:
            print(f"\n{'==' * 50}\n")
            print(f'Performing Round: {rounds:0>2}')
            responses = llm.generate(open_threads, sampling_params=sampling_params)
            open_threads = []
            for resp in responses:
                old_input_text = resp.prompt
                for out in resp.outputs:
                    end_reason = out.stop_reason
                    
                    if end_reason is None and out.finish_reason == 'length':
                        end_reason = 'length'
                    
                    output_text = out.text
                    next_input_text, final_answer, error_name, error_msg = post_processing_v2(output_text, old_input_text, end_reason)

                    if final_answer is not None:
                        resp_details = {
                            'solution': next_input_text,
                            'final_answer': final_answer,
                            'error_name': error_name,
                            'error_msg': error_msg,
                        }
                        closed_threads.append(resp_details)
                        final_answers.append(final_answer)
                    elif error_name is not None:
                        resp_details = {
                            'solution': next_input_text,
                            'final_answer': final_answer,
                            'error_name': error_name,
                            'error_msg': error_msg,
                        }
                        closed_threads.append(resp_details)
                    else:
                        assert old_input_text != next_input_text
                        if len(tokenizer.tokenize(next_input_text)) < 4096:
                            open_threads.append(next_input_text)

            open_threads = list(set(open_threads))
            print(f"Total Closed Threads: {len(closed_threads)}")
            print(f"Current Open Threads: {len(open_threads)}")
            print(f"Top 3 Answers       : {Counter(final_answers).most_common(n=3)}")
            rounds += 1
        
        with jsonlines.open(save_fpath, mode='w') as f:
            f.write_all(closed_threads)

    fa, nthreads = Counter(final_answers).most_common(n=1)[0]
    print(f"\nOur Final Answer      : {fa}")
    print(f"Original Final Answer : {ofa}")
    print(f"Is OFA in Top3 Answers: {ofa in [str(x) for x, _  in Counter(final_answers).most_common(n=3)]}")
    print(f"\n{'++' * 50}\n")

    if ofa == fa:
        ncorrect += 1
    else:
        nwrong += 1
    
    if ofa in [x for x, _  in Counter(final_answers).most_common(n=3)]:
        ncorrect_top3 += 1

    print(f"Running Ledger:\n\tTotal Problems Attempted\t: {nattempted}\n\tProblems Solved Correctly\t: {ncorrect}\n\tProblems Solved Incorrectly\t: {nwrong}\n\tProblems Solved Correctly (Top3): {ncorrect_top3}")
    print(f"\n{'++' * 50}\n")

  0%|          | 0/2883 [00:00<?, ?it/s]

Exp-ID: Exp0000
Problem: How many vertical asymptotes does the graph of $y=\frac{2}{x^2+x-6}$ have?
Original Final Answer: 2

Our Final Answer      : 2
Original Final Answer : 2
Is OFA in Top3 Answers: True

++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

Running Ledger:
	Total Problems Attempted	: 1
	Problems Solved Correctly	: 1
	Problems Solved Incorrectly	: 0
	Problems Solved Correctly (Top3): 1

++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

Exp-ID: Exp0001
Problem: What is the positive difference between $120\%$ of 30 and $130\%$ of 20?
Original Final Answer: 10

Our Final Answer      : 10
Original Final Answer : 10
Is OFA in Top3 Answers: True

++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

Running Ledger:
	Total Problems Attempted	: 2
	Problems Solved Correctly	: 2
	Problems Solved Incorrectly	: 0
	Problems Solved Co


Processed prompts: 100%|██████████| 1/1 [00:15<00:00, 15.10s/it, Generation Speed: 282.21 toks/s]


Total Closed Threads: 1
Current Open Threads: 4
Top 3 Answers       : [('125', 1)]


Performing Round: 02



Processed prompts: 100%|██████████| 4/4 [00:23<00:00,  5.89s/it, Generation Speed: 868.99 toks/s]


...

Total Closed Threads: 187
Current Open Threads: 6
Top 3 Answers       : [('5', 187)]


Performing Round: 26



Processed prompts: 100%|██████████| 6/6 [00:03<00:00,  1.58it/s, Generation Speed: 725.59 toks/s]


Total Closed Threads: 187
Current Open Threads: 6
Top 3 Answers       : [('5', 187)]


Performing Round: 27



Processed prompts: 100%|██████████| 6/6 [00:03<00:00,  1.55it/s, Generation Speed: 712.54 toks/s]


KeyboardInterrupt: 

In [17]:
print(open_threads[0])

Your are a high school student appearing for your math exam.
You will be given a math problem to solve, and your job is to write the detailed step by step solution, with good mathematical reasoning and using python code for calculations, simplifications, solving equations, etc.

Format help and instructions:
- Problem has been given enclosed in <math_problem></math_problem> tags.
- Every step must be written within <start_of_step><end_of_step> tags.
- Code needs to be always written inside <code_block></code_block> tags - No backticks or triple backticks.
- Code output will be given in <code_output_block></code_output_block> tags

Every calculation should be performed using writing supporting python code.

<math_problem>What is the greatest prime factor of the sum of the arithmetic sequence $1 + 2 + 3 + \cdots + 80$?</math_problem>
<start_of_solution>
<start_of_step>1
The sum of an arithmetic sequence can be found using the formula $S = \frac{n}{2}(a_1 + a_n)$, where $S$ is the sum, $n